In [1]:
""" Imports """
%load_ext autoreload
%autoreload 2
import json
from collections import defaultdict
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
from matplotlib.colors import hex2color, to_hex
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, mean_squared_error

from simulation_encoder.loaders.loader_retrieval import load_loaders
from simulation_encoder.models.model_retrieval import load_models


In [2]:
""" Image functions """
def plot_image_grid(images_data, fig, axes):
    for item in images_data:
        row = item['row']
        col = item['col']
        image = item['image']
        title = item.get('title', '')
        cmap = item.get('cmap', 'gray')
        
        if axes.ndim == 1:
            ax = axes[col]
        else:
            ax = axes[row, col]

        if title:
            ax.set_title(title, pad=0, fontsize=14)
            
        if isinstance(image, torch.Tensor):
            image = image.detach().cpu().numpy()
            
        ax.imshow(image, cmap=cmap)
        ax.axis('off')

def get_original_images(samples, labels, channels, start_row, title="Original Timepoint"):
    images_data = []
    for j, sample in enumerate(samples):
        images = sample.squeeze()
        
        if images.ndim == 2:
            images_data.append({
                'image': images,
                'title': f"{title} {labels[j]}",
                'row': start_row,
                'col': j
            })
        elif images.ndim == 3:
            for i, image in enumerate(images):
                title = f"{title} {labels[j]}" if i == 0 else ""
                images_data.append({
                    'image': image,
                    'title': title,
                    'row': start_row + i,
                    'col': j
                })
        else:
            raise ValueError("Unsupported image dimensions. Expected 2D or 3D images.")
            
    return images_data

def get_reconstructed_images(samples, model, channels, start_row, title="Predicted Timepoint"):
    images_data = []
    for j, sample in enumerate(samples):
        model.eval()
        sample_input = sample.unsqueeze(0)
        
        with torch.no_grad():
            result = model(sample_input)
            
        reconstructed_image, label_pred = result[:2]
        reconstructed_image = reconstructed_image.squeeze()
        label_pred = torch.max(label_pred, dim=1)[1].item()
        
        if reconstructed_image.ndim == 2:
            images_data.append({
                'image': reconstructed_image,
                'title': f"{title}",
                'row': start_row,
                'col': j
            })
        elif reconstructed_image.ndim == 3:
            for i, image in enumerate(reconstructed_image):
                title = f"{title} {label_pred}" if i == 0 else ""
                images_data.append({
                    'image': image,
                    'title': title,
                    'row': start_row + i,
                    'col': j
                })
        else:
            raise ValueError("Unsupported image dimensions. Expected 2D or 3D images.")
            
    return images_data

def get_difference_images(samples, model, channels, start_row, cmap="bwr"):
    imgs = []
    for j, sample in enumerate(samples):
        with torch.no_grad():
            recon, _ = model(sample.unsqueeze(0))[:2]
        recon, orig = recon.squeeze(), sample.squeeze()

        diffs = recon - orig
        if diffs.ndim == 2:
            diffs = diffs.unsqueeze(0)

        for ch_idx, diff in enumerate(diffs):
            m = diff.abs().max().item() or 1.0
            imgs.append(
                {
                    "image": diff,
                    "title": "",
                    "row": start_row + ch_idx,
                    "col": j,
                    "cmap": cmap,
                    "vmin": -m,
                    "vmax": m,
                }
            )
    return imgs

def get_saliency_maps(dataset, indices, model, start_row, title="Saliency Map"):
    images_data = []
    for j, idx in enumerate(indices):
        data = dataset[idx]
        input_tensor = data[0].unsqueeze(0)
        
        # Generate saliency map
        saliency_map = model.get_saliency_map(input_tensor).squeeze()
        
        images_data.append({
            'image': saliency_map,
            'title': f"{title}",
            'row': start_row,
            'col': j,
            'cmap': 'hot'
        })
        
    return images_data

""" Plotting functions """
def extract_seeds(all_loaders, loader_type):
    return {
        experiment: {
            dataset_name: [key.split('_')[-1] for key in datasets["full"].get_seed_keys(loader_type)]
            for dataset_name, datasets in models.items()
        }
        for experiment, models in all_loaders.items()
    }

def process_colors(experiment, dataset_name, seeds, timepoints):
    color_seeds = [int(seed) for seed in seeds[experiment][dataset_name]]
    color_timepoints = [int(timepoint) for timepoint in timepoints[experiment][dataset_name]]
    color_seeds_unique = list(set(color_seeds))
    return color_seeds, color_timepoints, color_seeds_unique

def plot_time_ordered_points(ax, embeddings, color_seeds, color_timepoints, title, xlabel, ylabel):
    selected_points = [(x, y) for k, (x, y) in enumerate(embeddings) if color_seeds[k] in color_seeds_unique]
    selected_timepoints = [color_timepoints[k] for k in range(len(color_seeds)) if color_seeds[k] in color_seeds_unique]

    if len(selected_points) > 1:
        # Ensure points are ordered by time
        sorted_indices = sorted(range(len(selected_timepoints)), key=lambda k: selected_timepoints[k])
        ordered_points = [selected_points[i] for i in sorted_indices]
        ordered_timepoints = [selected_timepoints[i] for i in sorted_indices]

        # Create line segments between consecutive points
        segments = [
            (ordered_points[i], ordered_points[i + 1]) for i in range(len(ordered_points) - 1)
        ]
        segment_colors = ordered_timepoints[:-1]
        norm = plt.Normalize(min(segment_colors), max(segment_colors))
        lc = LineCollection(segments, cmap='viridis', norm=norm)
        lc.set_array(segment_colors)
        lc.set_linewidth(2)

        ax.add_collection(lc)
        ax.scatter(*zip(*ordered_points), c=ordered_timepoints, cmap='viridis', s=50)  # Optional to highlight points

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.colorbar(lc, ax=ax, label='Timepoints')
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Data Loading

In [3]:
study_name = "full-gastruloid-no-time"
image_dir = "data"
num_timepoints = 9

results_dir = f"results/{study_name}"

all_models = load_models(results_dir, num_timepoints=num_timepoints, best_models_flag=True)
all_loaders = load_loaders(results_dir, image_dir)

all_samples = {}
all_labels = {}
data_loaders = {}
random_indices = {}

for dataset_name, loader in all_loaders.items():
    data_loaders[dataset_name] = {
        "train": loader.get_dataloader('train'),
        "test": loader.get_dataloader('test'),
        "full": loader,
    }
    
    loader_test_len = len(data_loaders[dataset_name]["test"].dataset)
    
    if dataset_name not in random_indices:
        random_indices[dataset_name] = np.random.choice(loader_test_len, 5, replace=False)
    
    indices = random_indices[dataset_name]
    all_samples[dataset_name] = []
    all_labels[dataset_name] = []
    
    for i in indices:
        image, label = data_loaders[dataset_name]["test"].dataset[i]
        sample_id = loader._get_data_feature(i, "sample_id")
        array_num = int(sample_id.split('_')[0][-1])
        all_samples[dataset_name].append(image)
        all_labels[dataset_name].append("Euploid" if array_num == 1 else "Anueploid")


all_models = dict(sorted(all_models.items()))

print(f"Models: {list(all_models.keys())}")
print(f"Datasets: {list(all_loaders.keys())}")

Loading model neuralop_medium (3.41e+07 parameters)
Loading model cae_medium (8.70e+06 parameters)


IndexError: list index out of range

## Image reconstruction

In [ ]:
for model_name, dataset in all_models.items():
    for dataset_name, model in dataset.items():
        print(model)
        samples = all_samples[dataset_name]
        labels = all_labels[dataset_name]
        test_dataset = data_loaders[dataset_name]["test"].dataset
        channels = data_loaders[dataset_name]["full"].channels
        indices = random_indices[dataset_name]
        
        num_channels = len(channels)
        nrows = num_channels * 3 + 1  
        fig, axes = plt.subplots(nrows=nrows, ncols=5, figsize=(15, 12))
        
        all_images_data = []
        
        original_images = get_original_images(samples, labels, channels, start_row=0)
        all_images_data.extend(original_images)
    
        recon_start_row = num_channels
        reconstructed_images = get_reconstructed_images(samples, model, channels, start_row=recon_start_row, title="")
        all_images_data.extend(reconstructed_images)
        
        diff_start_row = num_channels * 2
        difference_images = get_difference_images(
            samples, model, channels, start_row=diff_start_row
        )
        all_images_data.extend(difference_images)

        saliency_start_row = num_channels * 3
        saliency_maps = get_saliency_maps(
            test_dataset, indices, model, start_row=saliency_start_row
        )
        all_images_data.extend(saliency_maps)

        # for i in range(len(indices)):
        #     plt.imsave(f"figures/gastruloid/gastruloid_original_{model_name}_{i}.png", all_images_data[i]['image'], cmap='gray')
        #     plt.imsave(f"figures/gastruloid/gastruloid_reconstructed_{model_name}_{i}.png", all_images_data[i + 5]['image'], cmap='gray')
        #     plt.imsave(f"figures/gastruloid/gastruloid_difference_{model_name}_{i}.png", all_images_data[i + 10]['image'], cmap='bwr')
            
        plot_image_grid(all_images_data, fig, axes)
        fig.suptitle(f"Model: {model_name} Dataset: {dataset_name}", fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

In [ ]:
# for dataset_name, loader in all_loaders.items():
#     data_loaders[dataset_name] = {
#         "train": loader.get_dataloader('train'),
#         "test": loader.get_dataloader('test'),
#         "full": loader,
#     }
    
#     all_samples[dataset_name] = []
#     test_dataset = data_loaders[dataset_name]["test"].dataset
#     for i in range(len(test_dataset)):
#         sample_id = loader._get_data_feature(i, "sample_id")
#         if sample_id == "array1_212":
#             encoding, _ = test_dataset[i]
#             timepoint = loader._get_data_feature(i, "timepoint")
#             # Save both the encoding and the timepoint
#             all_samples[dataset_name].append({'encoding': encoding, 'timepoint': timepoint})

# for model_name, dataset in all_models.items():
#     if model_name != "neuralop_16":
#         continue
#     for dataset_name, model in dataset.items():
#         sample_entries = all_samples.get(dataset_name, [])
#         if not sample_entries:
#             continue
#         channels = data_loaders[dataset_name]["full"].channels
        
#         for idx, entry in enumerate(sample_entries):
#             encoding = entry['encoding']
#             timepoint = entry['timepoint']
            
#             # Pass a title that includes the timepoint to the function.
#             reconstructed_images = get_reconstructed_images(
#                 [encoding], model, channels, start_row=0, title=f"Timepoint: {timepoint}"
#             )
            
#             # Plot each reconstructed image individually.
#             for img_dict in reconstructed_images:
#                 fig, ax = plt.subplots(figsize=(5, 5))
#                 ax.imshow(img_dict['image'], cmap='gray')
#                 ax.axis('off')
#                 plt.savefig(f"reconstructed_image_{model_name}_{dataset_name}_{img_dict['title']}.png")

In [ ]:
# activations = {}

# # Capture the FNO layer output using a forward hook
# def fno_hook(module, input, output):
#     activations['fno'] = output.detach()

# for model_name, dataset in all_models.items():
#     for dataset_name, model in dataset.items():
#         print(f"Processing Model: {model_name} | Dataset: {dataset_name}")

#         fno_layer = None
#         for module_name, module in model.named_modules():
#             if "FNO" in str(type(module)):
#                 fno_layer = module
#                 print(f"Found FNO layer: {module_name}")
#                 break
        
#         if fno_layer is None:
#             print(f"FNO layer not found in Model: {model_name}, Dataset: {dataset_name}")
#             continue
        
#         hook_handle = fno_layer.register_forward_hook(fno_hook)
        
#         samples = all_samples[dataset_name]
#         labels = all_labels[dataset_name]
        
#         model.eval()
        
#         fig, axes = plt.subplots(2, len(samples), figsize=(15, 6))
#         if len(samples) == 1:
#             axes = np.array(axes).reshape(2, 1)
        
#         for idx, sample in enumerate(samples):
#             sample = sample['encoding']
#             sample_tensor = sample.unsqueeze(0)
            
#             activations.clear()
#             _ = model(sample_tensor)
#             fno_output = activations.get('fno')

#             original_img = sample.squeeze().cpu().numpy() if sample.dim() == 3 else sample.cpu().numpy()
#             axes[0, idx].imshow(original_img, cmap='gray')
#             axes[0, idx].set_title(f"Label: {labels[idx]}")
#             axes[0, idx].axis('off')
            
#             if fno_output is not None:
#                 activation_img = fno_output[0, 0].cpu().numpy()
#                 axes[1, idx].imshow(activation_img, cmap='viridis')
#                 axes[1, idx].set_title("FNO Activation")
#                 axes[1, idx].axis('off')
#             else:
#                 axes[1, idx].text(0.5, 0.5, 'No activation', ha='center', va='center')
#                 axes[1, idx].axis('off')
        
#         fig.suptitle(f"Original Image and FNO Activation\nModel: {model_name}, Dataset: {dataset_name}", fontsize=16)
#         plt.tight_layout(rect=[0, 0, 1, 0.95])
#         plt.show()

#         hook_handle.remove()

In [ ]:
random_images = {}
random_reconstructed_images = {}
processed_datasets = {}

for model_name, datasets in all_models.items():
    random_reconstructed_images[model_name] = {}
    for dataset_name, model in datasets.items():
        if dataset_name in processed_datasets:
            random_images[dataset_name] = processed_datasets[dataset_name]['random_images']
            random_reconstructed_images[dataset_name] = processed_datasets[dataset_name]['random_reconstructed_images']
        else:
            train_loader = data_loaders[dataset_name]["train"]
            test_loader = data_loaders[dataset_name]["test"]

            encoder = model.encoder
            last_layer = encoder[-1]
            dim = last_layer.out_features

            
            train_images = [
                np.concatenate([img.flatten() for img in batch[0]])
                for batch in train_loader
            ]

            test_images = [
                np.concatenate([img.flatten() for img in batch[0]])
                for batch in test_loader
            ]
            
            pca = PCA(n_components=dim)
            train_images_pca = pca.fit_transform(train_images)
            test_images_pca = pca.transform(test_images)
            reconstructed_test_images = pca.inverse_transform(test_images_pca)

            indices = random_indices[dataset_name]
            random_images_list = [test_images[i] for i in indices]
            random_reconstructed_images_list = [reconstructed_test_images[i] for i in indices]

            processed_datasets[dataset_name] = {
                'random_images': random_images_list,
                'random_reconstructed_images': random_reconstructed_images_list,
            }
            
            random_images[dataset_name] = random_images_list
            random_reconstructed_images[dataset_name] = random_reconstructed_images_list

In [ ]:
for dataset, _ in processed_datasets.items():
    reshaped_random_images = [
        img.reshape(num_channels, 128, 128) for img in random_images[dataset]
    ]
    reshaped_random_reconstructed_images = [
        img.reshape(num_channels, 128, 128) for img in random_reconstructed_images[dataset]
    ]

    nrows = num_channels * 2
    fig, axes = plt.subplots(nrows=nrows, ncols=5, figsize=(15, 10))
    
    for j, sample in enumerate(reshaped_random_images):
        for i, image in enumerate(sample):
            axes[i, j].imshow(image, cmap='gray')
            axes[i, j].axis('off')

    for j, sample in enumerate(reshaped_random_reconstructed_images):
        for i, image in enumerate(sample):
            axes[num_channels + i, j].imshow(image, cmap='gray')
            axes[num_channels + i, j].axis('off')

    fig.suptitle(f"PCA Compression for {dataset}")
    plt.show()

## Traditional dimension reduction techniques

In [ ]:
processed_pca = {}

for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():
        if dataset_name in processed_pca:
            continue  

        train_loader = data_loaders[dataset_name]["train"]
        test_loader = data_loaders[dataset_name]["test"]

        def flatten_images(loader):
            flat_images = []
            for batch in loader:
                imgs_batch = batch[0].squeeze()
                flat_imgs = [img.flatten() for img in imgs_batch]
                flat_images.append(np.concatenate(flat_imgs))
            return flat_images

        train_images = flatten_images(train_loader)
        test_images = flatten_images(test_loader)
        test_timepoints = [batch[1].item() for batch in test_loader]

        pca = PCA(n_components=2)
        train_images_pca = pca.fit_transform(train_images)
        test_images_pca = pca.transform(test_images)

        def plot_projection(projection, title):
            plt.scatter(projection[:, 0], projection[:, 1], label="test", c=test_timepoints, s=10, cmap="viridis")
            plt.tight_layout()
            plt.title(title)
            plt.show()

        plot_projection(test_images_pca, f"PCA of test data - {dataset_name}")

    break

## Visualization of the latent space

In [ ]:
timepoints = defaultdict(lambda: defaultdict(list))

for model_name, dataset in all_models.items():
    for dataset_name, model in dataset.items():
        
        loader = data_loaders[dataset_name]["full"]
        for idx in loader._test_indices:
            timepoints[model_name][dataset_name].append(int(loader._get_data_feature(idx, 'timepoint')))

In [ ]:
nn_embeddings = {}

for model_name, datasets in all_models.items():
    nn_embeddings[model_name] = {}
    for dataset_name, model in datasets.items():
        nn_embeddings[model_name][dataset_name] = {}
        print(model_name, dataset_name)
            
        test_loader = data_loaders[dataset_name]["test"]
        num_dimensions = model.latent_dim

        model.eval()
        with torch.no_grad():
            encoded_samples = []
            for inputs, _ in test_loader:
                encoded_batch = model.encode(inputs)
                encoded_samples.append(encoded_batch)
            encoded_samples = torch.cat(encoded_samples, dim=0)

            mean = encoded_samples.mean(dim=0, keepdim=True)
            std = encoded_samples.std(dim=0, keepdim=True)
            normalized_embeddings = (encoded_samples - mean) / (std + 1e-8)

            nn_embeddings[model_name][dataset_name] = normalized_embeddings

In [ ]:
pca_embeddings = {}
for model_name, dataset in nn_embeddings.items():
    pca_embeddings[model_name] = {}
    for dataset_name, embeddings in dataset.items():
        try:
            pca = PCA()
            pca_embeddings[model_name][dataset_name] = pca.fit_transform(embeddings)

            print(f"{model_name} | {dataset_name} - Explained Variance: {pca.explained_variance_ratio_}")
        except ValueError:
            print(f"Skipping {model_name} PCA")

In [ ]:
for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():
        print(model_name)
        pca_embedding = pca_embeddings[model_name][dataset_name]
        times = [int(t) for t in timepoints[model_name][dataset_name]]

        df_plot = pd.DataFrame({
            "PCA_1": pca_embedding[:, 0],
            "PCA_2": pca_embedding[:, 1],
            "Timepoint": times
        })

        plt.figure(figsize=(8, 8))

        norm = mcolors.Normalize(vmin=min(times), vmax=max(times))

        cmap = plt.colormaps["gray_r"]
        truncated_cmap = mcolors.LinearSegmentedColormap.from_list(
            "trunc_gray", cmap(np.linspace(0.1, 1.0, 256))
        )

        scatter = plt.scatter(
            df_plot["PCA_1"], df_plot["PCA_2"],
            c=df_plot["Timepoint"],
            cmap=truncated_cmap,
            norm=norm,
            s=50
        )

        plt.grid(False)
        plt.xticks([])
        plt.yticks([])
        plt.xlabel('')
        plt.ylabel('')
        plt.title('')

        ax = plt.gca()
        for spine in ax.spines.values():
            spine.set_visible(False)

        plt.tight_layout()
        plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(1.5, 6))

sm = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
sm.set_array([])

cbar = plt.colorbar(sm, cax=ax)

cbar.ax.tick_params(left=False, right=False, labelleft=False)
cbar.ax.set_yticklabels([])

plt.tight_layout()
plt.show()

In [ ]:
processed_pca = {}

for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():
        if dataset_name in processed_pca:
            continue
        else:
            processed_pca[dataset_name] = {}

            train_loader = data_loaders[dataset_name]["train"]
            test_loader = data_loaders[dataset_name]["test"]

            encoder = model.encoder
            last_layer = encoder[-1]
            dim = last_layer.out_features
            
            train_images = [
                np.concatenate([img.flatten() for img in batch[0]])
                for batch in train_loader
            ]

            test_images = [
                np.concatenate([img.flatten() for img in batch[0]])
                for batch in test_loader
            ]
        
            pca = PCA(n_components=dim)
            train_pca_images = pca.fit_transform(train_images)
            reconstructed_train_images = pca.inverse_transform(train_pca_images)

            test_pca_images = pca.transform(test_images)
            reconstructed_test_images = pca.inverse_transform(test_pca_images)

            train_mse = mean_squared_error(train_images, reconstructed_train_images)
            test_mse = mean_squared_error(test_images, reconstructed_test_images)

            processed_pca[dataset_name]["train"] = train_mse
            processed_pca[dataset_name]["test"] = test_mse

In [ ]:
nn_reconstructions = {}

# for model_name, dataset in all_models.items():
#     nn_reconstructions[model_name] = {}
#     for dataset_name, model in dataset.items():
#         print(f'{model_name} | {dataset_name}')

#         embeddings = nn_embeddings[model_name][dataset_name]
#         model.eval()
#         with torch.no_grad():
#             reconstructed_samples = model.decode_image(embeddings)
#         reconstructed_samples = reconstructed_samples.reshape(-1, 128 * 128)
        
#         test_loader = data_loaders[dataset_name]["test"]
#         test_images = np.concatenate(
#             [batch[0].numpy().reshape(len(batch[0]), -1) for batch in test_loader],
#             axis=0
#         )
        
#         test_mse = mean_squared_error(test_images, reconstructed_samples)

#         nn_reconstructions[model_name][dataset_name] = test_mse

# print("Reconstruction MSE")
# print(f"{'Model':<15} {'Dataset':<20} {'Test MSE':<15}")
# print("-" * 65)
# for dataset_name, mse in processed_pca.items():
#     print(f"{'PCA':<20} {dataset_name:<20} {mse['test']:<15.6f}")

# for model_name, model_results in nn_reconstructions.items():
#     for dataset_name, mse in model_results.items():
#         print(f"{model_name:<20} {dataset_name:<20} {mse:<15.6f}")

## Pairwise latent dimension visualization

In [ ]:
# for model_name, dataset in nn_embeddings.items():
#     for dataset_name, embeddings in dataset.items():
#         num_dimensions = embeddings.shape[1]
            
#         if num_dimensions >= 32:
#             print(f"Skipping {model_name} | {dataset_name} due to large latent space size ({num_dimensions} dimensions)")
#             continue

#         df = pd.DataFrame(embeddings, columns=[f"Dimension {j+1}" for j in range(num_dimensions)])
#         df["timepoint"] = timepoints[model_name][dataset_name]
        
#         print(f"Plotting for {model_name} | {dataset_name}")
        
#         g = sns.pairplot(df, hue="timepoint", palette="viridis", plot_kws={"s": 15}, corner=True)
#         g._legend.remove()
#         plt.tight_layout()
#         plt.show()

In [ ]:
for model_name, dataset in nn_embeddings.items():
    for dataset_name, embeddings in dataset.items():
        num_dimensions = embeddings.shape[1]
            
        if num_dimensions > 32:
            print(f"Skipping {model_name} | {dataset_name} due to large latent space size ({num_dimensions} dimensions)")
            continue

        print(f"Plotting distributions for {model_name} | {dataset_name}")

        cols = 4
        rows = int(np.ceil(num_dimensions / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
        axes = axes.flatten()

        for dim_idx in range(num_dimensions):
            sns.histplot(
                embeddings[:, dim_idx],
                ax=axes[dim_idx],
                kde=True,            
                bins="auto",        
            )
            axes[dim_idx].set_title(f"Dim {dim_idx + 1}")
            axes[dim_idx].set_xlabel("")
            axes[dim_idx].set_ylabel("Freq")

        for ax in axes[num_dimensions:]:
            ax.remove()

        plt.tight_layout()
        plt.show()

In [ ]:
"""Scan through real gastruloid latent space"""
# for model_name, dataset in all_models.items():
#     for dataset_name, model in dataset.items():
#         embeddings = nn_embeddings[model_name][dataset_name]
#         loader = data_loaders[dataset_name]["full"]
#         num_dimensions = embeddings.shape[1]

#         test_indices = loader._test_indices
#         sample_ids = []
#         dimension_data = [] # List to hold (sample_id, embedding_vector)
#         for idx in test_indices:
#             sample_id = loader._get_data_feature(idx, 'sample_id')
#             sample_ids.append(sample_id)

#         for test_idx in range(len(test_indices)):
#             full_dataset_idx = test_indices[test_idx]
#             embedding_vector = embeddings[test_idx]
#             sample_id = loader._get_data_feature(full_dataset_idx, 'sample_id')
#             dimension_data.append({'sample_id': sample_id, 'embedding': embedding_vector})

#         for dim_idx in range(num_dimensions):
#             dimension_values = sorted(dimension_data, key=lambda x: x["embedding"][dim_idx])

#             min_sample = dimension_values[0]
#             max_sample = dimension_values[-1]
#             median_idx = len(dimension_values) // 2
#             median_sample = dimension_values[median_idx]

#             fig, axes = plt.subplots(1, 3, figsize=(6,6))
#             for test_idx in range(len(test_indices)):
#                 full_dataset_idx = test_indices[test_idx]
#                 sample_id = loader._get_data_feature(full_dataset_idx, 'sample_id')
#                 if sample_id == min_sample["sample_id"]:
#                     image, label = data_loaders[dataset_name]["test"].dataset[test_idx]
#                     if label == 8:
#                         axes[0].imshow(image.squeeze(), cmap="gray")
#                         axes[0].axis('off')
#                         axes[0].set_title(f"{sample_id}")
#                 if sample_id == median_sample["sample_id"]:
#                     image, label = data_loaders[dataset_name]["test"].dataset[test_idx]
#                     if label == 8:
#                         axes[1].imshow(image.squeeze(), cmap="gray")
#                         axes[1].axis('off')
#                         axes[1].set_title(f"{sample_id}")
#                 if sample_id == max_sample["sample_id"]:
#                     image, label = data_loaders[dataset_name]["test"].dataset[test_idx]
#                     if label == 8:
#                         axes[2].imshow(image.squeeze(), cmap="gray")
#                         axes[2].axis('off')
#                         axes[2].set_title(f"{sample_id}")

#             plt.tight_layout()
#             plt.show()
        

In [ ]:
sample_ids = defaultdict(lambda: defaultdict(list))

for model_name, dataset in all_models.items():
    for dataset_name, model in dataset.items():
        
        loader = data_loaders[dataset_name]["full"]
        for i, idx in enumerate(loader._test_indices):
            sample_id = loader._get_data_feature(idx, 'sample_id')
            sample_ids[model_name][dataset_name].append(sample_id)

In [ ]:
base_colors = {1: "#696767", 2: "#083566"}

for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():

        pca_embedding = pca_embeddings[model_name][dataset_name]
        current_sample_ids = sample_ids[model_name][dataset_name]

        array1_indices = [i for i, sid in enumerate(current_sample_ids) if sid.startswith('array1')]
        array2_indices = [i for i, sid in enumerate(current_sample_ids) if sid.startswith('array2')]

        fig, ax = plt.subplots(figsize=(8, 8))
        sns.scatterplot(x=pca_embedding[array1_indices, 0], y=pca_embedding[array1_indices, 1],
            ax=ax, color=base_colors[1], s=15, label='Euploid', edgecolor='none')
        sns.scatterplot(x=pca_embedding[array2_indices, 0], y=pca_embedding[array2_indices, 1],
            ax=ax, color=base_colors[2], s=15, label='Aneuploid', edgecolor='none')

        ax.grid(False)
        ax.set_title(f'{model_name} | {dataset_name} condition', fontsize=16, y=1.02)
        ax.set_xlabel('PCA Dimension 1')
        ax.set_ylabel('PCA Dimension 2')

        ax.legend(loc='upper right')

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

In [ ]:
def get_shaded_color(base_hex, timepoint, min_tp, max_tp, light_factor=0.2):
    base_rgb = hex2color(base_hex)
    if max_tp == min_tp:
        normalized_tp = 1.0
    else:
        normalized_tp = (timepoint - min_tp) / (max_tp - min_tp)
    interpolation_amount = light_factor + normalized_tp * (1 - light_factor)

    shaded_rgb = [1.0 * (1 - interpolation_amount) + c * interpolation_amount for c in base_rgb]
    shaded_rgb = [max(0, min(1, c)) for c in shaded_rgb]
    return shaded_rgb

for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():

        pca_embedding = pca_embeddings[model_name][dataset_name]
        current_sample_ids = sample_ids[model_name][dataset_name]
        current_timepoints = timepoints[model_name][dataset_name]

        min_timepoint = min(current_timepoints)
        max_timepoint = max(current_timepoints)

        point_colors = []
        arrays_in_plot = set()

        for i in range(len(pca_embedding)):
            sid = current_sample_ids[i]
            tp = current_timepoints[i]

            base_hex = None
            if sid.startswith(('array1', 'array3', 'array5')):
                base_hex = base_colors[1]
                arrays_in_plot.add(1)
            elif sid.startswith(('array2', 'array4', 'array6')):
                base_hex = base_colors[2]
                arrays_in_plot.add(2)

            shaded_color_rgb = get_shaded_color(base_hex, tp, min_timepoint, max_timepoint)
            point_colors.append(shaded_color_rgb)

        fig, ax = plt.subplots(figsize=(8, 8))
        sns.scatterplot(x=pca_embedding[:, 0], y=pca_embedding[:, 1],
                        ax=ax,
                        c=point_colors,
                        s=15, edgecolor='none')

        ax.grid(False)

        ax.set_title(f'{model_name} | {dataset_name} condition', fontsize=16, y=1.02)
        ax.set_xlabel('PCA Dimension 1')
        ax.set_ylabel('PCA Dimension 2')

        legend_handles = []
        if 1 in arrays_in_plot:
            legend_handles.append(mpatches.Patch(color=base_colors[1], label='Euploid'))
        if 2 in arrays_in_plot:
            legend_handles.append(mpatches.Patch(color=base_colors[2], label='Aneuploid'))

        ax.legend(handles=legend_handles, loc='upper right')
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()


In [ ]:
for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():
        emb = pca_embeddings[model_name][dataset_name]
        sids = sample_ids[model_name][dataset_name]
        tps = timepoints[model_name][dataset_name]
        t_min, t_max = min(tps), max(tps)

        colors, traj = [], defaultdict(list)
        for i, (sid, tp) in enumerate(zip(sids, tps)):
            base = 1 if sid.startswith(('array1', 'array3', 'array5')) else \
                   2 if sid.startswith(('array2', 'array4', 'array6')) else None
            if base:
                color = get_shaded_color(base_colors[base], tp, t_min, t_max)
                traj[sid].append((tp, emb[i], color))
            else:
                color = "gray"
            colors.append(color)

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.scatter(emb[:, 0], emb[:, 1], c=colors, s=15, edgecolor='none')

        for entries in traj.values():
            entries.sort(key=lambda x: x[0])
            for (t1, e1, c1), (t2, e2, _) in zip(entries, entries[1:]):
                ax.plot([e1[0], e2[0]], [e1[1], e2[1]], color=c1,
                        linestyle='--', linewidth=2.0, alpha=0.4)

        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(''); ax.set_ylabel('')
        ax.set_title('')
        ax.grid(False)
        for spine in ax.spines.values():
            spine.set_visible(False)

        plt.tight_layout(pad=0)
        plt.savefig(f"figures/gastruloid/gastruloid_{model_name}_combined_trajectories.png", dpi=300, bbox_inches='tight', pad_inches=0)
        plt.show()

In [ ]:
classification_file = f"results_downstream/{study_name}_classification_results.json"
incorrect_samples = defaultdict(lambda: defaultdict(list))
try:
    with open(classification_file, "r") as f:
        classification_results = json.load(f)

    for item in classification_results["model_dataset_results"]:
        incorrect_samples[item["model_name"]][item["dataset_name"]] = item["incorrect_predictions"]

    for model_name, datasets in all_models.items():
        print(model_name)
        for dataset_name, model in datasets.items():
            emb = pca_embeddings[model_name][dataset_name]
            ids = sample_ids[model_name][dataset_name]
            tps = timepoints[model_name][dataset_name]
            incorrect = set(incorrect_samples[model_name][dataset_name])

            tmin, tmax = min(tps), max(tps)
            col = []; traj = defaultdict(list)

            for i, (sid, tp) in enumerate(zip(ids, tps)):
                base = 1 if sid.startswith(('array1', 'array3', 'array5')) else \
                       2 if sid.startswith(('array2', 'array4', 'array6')) else None
                if base:
                    color = get_shaded_color(base_colors[base], tp, tmin, tmax)
                    col.append(color)
                    traj[sid].append((tp, emb[i], color))
                else:
                    col.append("gray")

            fig, ax = plt.subplots(figsize=(8, 8))
            ax.scatter(emb[:, 0], emb[:, 1], c=col, s=25, edgecolor='none')

            for sid, entries in traj.items():
                entries.sort(key=lambda x: x[0])
                for (_, e1, c1), (_, e2, _) in zip(entries, entries[1:]):
                    ax.plot([e1[0], e2[0]], [e1[1], e2[1]],
                            color='red' if sid in incorrect else c1,
                            linestyle='-' if sid in incorrect else '--',
                            linewidth=2.0 if sid in incorrect else 2.0,
                            alpha=0.8 if sid in incorrect else 0.4)

            ax.set_xticks([]); ax.set_yticks([])
            ax.set_xlabel(''); ax.set_ylabel('')
            ax.set_title('')
            ax.grid(False)
            for spine in ax.spines.values():
                spine.set_visible(False)

            plt.tight_layout(pad=0)
            plt.savefig(f"figures/gastruloid/gastruloid_{model_name}_combined_trajectories_incorrect.png", dpi=300, bbox_inches='tight', pad_inches=0)
            plt.show()

except FileNotFoundError:
    print(f"[Warning] missing file: {classification_file}")
except Exception as e:
    print(f"[Error] {e}")

In [ ]:
for model_name, datasets in all_models.items():
    print(model_name)
    for dataset_name, model in datasets.items():
        emb = pca_embeddings[model_name][dataset_name]
        ids = sample_ids[model_name][dataset_name]
        tps = timepoints[model_name][dataset_name]
        incorrect = set(incorrect_samples[model_name][dataset_name])

        tmin, tmax = min(tps), max(tps)

        data_by_type = {1: {'emb': [], 'col': [], 'traj': defaultdict(list)},
                        2: {'emb': [], 'col': [], 'traj': defaultdict(list)}}

        for i, (sid, tp) in enumerate(zip(ids, tps)):
            base = 1 if sid.startswith(('array1', 'array3', 'array5')) else \
                   2 if sid.startswith(('array2', 'array4', 'array6')) else None
            if base:
                shade = get_shaded_color(base_colors[base], tp, tmin, tmax)
                data_by_type[base]['emb'].append(emb[i])
                data_by_type[base]['col'].append(shade)
                data_by_type[base]['traj'][sid].append((tp, emb[i], shade))

        for array_type, label in zip([1, 2], ['euploid', 'aneuploid']):
            emb_array = np.array(data_by_type[array_type]['emb'])
            col_array = data_by_type[array_type]['col']
            traj_dict = data_by_type[array_type]['traj']

            fig, ax = plt.subplots(figsize=(8, 8))
            ax.scatter(emb_array[:, 0], emb_array[:, 1], c=col_array, s=25, edgecolor='none')

            for sid, entries in traj_dict.items():
                entries.sort(key=lambda x: x[0])
                for (_, e1, c1), (_, e2, _) in zip(entries, entries[1:]):
                    ax.plot([e1[0], e2[0]], [e1[1], e2[1]],
                            color='red' if sid in incorrect else c1,
                            linestyle='-' if sid in incorrect else '--',
                            linewidth=2.0 if sid in incorrect else 2.0,
                            alpha=0.8 if sid in incorrect else 0.4)

            ax.set_xticks([]); ax.set_yticks([])
            ax.set_xlabel(''); ax.set_ylabel(''); ax.set_title('')
            ax.grid(False)
            for spine in ax.spines.values():
                spine.set_visible(False)

            plt.tight_layout(pad=0)
            plt.savefig(f"figures/gastruloid/gastruloid_{model_name}_{label}_trajectories.png",
                        dpi=300, bbox_inches='tight', pad_inches=0)
            plt.show()

In [ ]:
# target_array_num = 1
# target_raft_id = 346

target_array_num = 2
target_raft_id = 183

sns.set_theme(style="whitegrid")
sns.set_context("notebook", font_scale=1.2)
fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette("viridis", 10)
color_idx = 0
model_positions = {}

for model_name, datasets in all_models.items():
    for dataset_name in datasets.keys():

        current_sample_ids = sample_ids[model_name][dataset_name]
        current_timepoints = timepoints[model_name][dataset_name]
        pca_embedding = pca_embeddings[model_name][dataset_name]

        target_indices = [i for i, sid in enumerate(current_sample_ids) if sid.startswith(f"array{target_array_num}_{target_raft_id}")]

        if target_indices:
            filtered_pca_embedding = pca_embedding[target_indices, :]
            filtered_timepoints = np.array([current_timepoints[i] for i in target_indices])
            filtered_sample_ids = [current_sample_ids[i] for i in target_indices]

            filtered_df = pd.DataFrame({
                'PC_0': filtered_pca_embedding[:, 0],
                'PC_1': filtered_pca_embedding[:, 1],
                'timepoint': filtered_timepoints
            })

            filtered_df = filtered_df.sort_values('timepoint')

            if len(filtered_df) > 1:
                ax.plot(filtered_df['PC_0'], filtered_df['PC_1'],
                        linestyle='--',
                        linewidth=1.5,
                        color='gray',
                        alpha=0.7,
                        zorder=1)

                sns.scatterplot(
                    data=filtered_df,
                    x='PC_0',
                    y='PC_1',
                    hue='timepoint',
                    s=150,
                    palette='viridis',
                    legend='brief',
                    ax=ax,
                    zorder=2
                )

                start_point = filtered_df.iloc[0]
                end_point = filtered_df.iloc[-1]

                ax.annotate(f"t={int(start_point['timepoint'])}",
                            (start_point['PC_0'], start_point['PC_1']),
                            textcoords="offset points",
                            xytext=(-5, 15),
                            fontsize=12,
                            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

                ax.annotate(f"t={int(end_point['timepoint'])}",
                        (end_point['PC_0'], end_point['PC_1']),
                        textcoords="offset points",
                        xytext=(-5, 15),
                        fontsize=12,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

                model_positions[f"{model_name} | {dataset_name}"] = (end_point['PC_0'], end_point['PC_1'])


for label, (x, y) in model_positions.items():
    ax.annotate(label,
               (x, y),
               textcoords="offset points",
               xytext=(15, 0),
               fontsize=12,
               bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))


condition = "Euploid" if target_array_num == 1 else "Aneuploid"
ax.set_title(f"{condition} | Raft {target_raft_id} Trajectory", fontsize=16)
ax.set_xlabel("PCA Dimension 1", fontsize=16)
ax.set_ylabel("PCA Dimension 2", fontsize=16)

ax.grid(False)
plt.legend([],[], frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
target_pc_nums = [0, 1, 2, 4, 5, 12]
target_tp = 8
metric_cols = ['Area', 'Round', 'Perim.', 'XM', 'YM']
segmentations_df = pd.read_csv("data/gastruloid_processed_128/segmentations.csv")

def parse_filename(filename):
    _, array, raft, time = filename.split("_")
    array_num = int(array[5:])
    raft = int(raft)
    time = int(time.split(".")[0])
    return f"array{array_num}_{raft}", int(time)

image_lookup = {}
for _, row in segmentations_df.iterrows():
    array_raft, timepoint = parse_filename(row["Image"])
    if array_raft is not None:
        if array_raft not in image_lookup or timepoint > image_lookup[array_raft][1]:
            image_lookup[array_raft] = (row["Image"], timepoint)

In [ ]:
from scipy.stats import pearsonr

for model_name, datasets in all_models.items():
    for dataset_name in datasets:
        current_ids = sample_ids[model_name][dataset_name]
        current_times = timepoints[model_name][dataset_name]
        embedding = pca_embeddings[model_name][dataset_name]

        target_pc_values = []

        for i, sid in enumerate(current_ids):
            if sid in image_lookup and current_times[i] == target_tp:
                image_name, _ = image_lookup[sid]
                pc_values = [embedding[i, pc_num] for pc_num in target_pc_nums]
                target_pc_values.append([image_name] + pc_values)

        pc_columns = ["Image"] + [f"PC_{n}" for n in target_pc_nums]
        pc_df = pd.DataFrame(target_pc_values, columns=pc_columns)

        merged_df = pd.merge(segmentations_df, pc_df, on="Image")

        corr_matrix = pd.DataFrame(index=metric_cols, columns=[f"PC_{n}" for n in target_pc_nums])
        for pc_name in corr_matrix.columns:
            for metric in corr_matrix.index:
                if metric in merged_df.columns:
                    x = merged_df[pc_name]
                    y = merged_df[metric]
                    corr, _ = pearsonr(x, y)
                    corr_matrix.at[metric, pc_name] = corr

        corr_matrix = corr_matrix.astype(float)

        plt.figure(figsize=(len(corr_matrix.columns) * 1.5 + 4, 8))
        sns.heatmap(
            corr_matrix,
            annot=False,
            cmap="RdBu_r",
            center=0,
            linewidths=0.5,
            square=True
        )
        plt.title(f"Model: {model_name} | Dataset: {dataset_name} | Timepoint {target_tp}")
        plt.tight_layout()
        plt.show()

In [ ]:
from scipy.stats import pearsonr

for model_name, datasets in all_models.items():
    print(model_name)
    for dataset_name in datasets:
        current_ids = sample_ids[model_name][dataset_name]
        current_times = timepoints[model_name][dataset_name]
        embedding = pca_embeddings[model_name][dataset_name]

        target_pc_values = []

        for i, sid in enumerate(current_ids):
            if sid in image_lookup and current_times[i] == target_tp:
                image_name, _ = image_lookup[sid]
                pc_values = [embedding[i, pc_num] for pc_num in target_pc_nums]
                target_pc_values.append([image_name] + pc_values)

        pc_columns = ["Image"] + [f"PC_{n}" for n in target_pc_nums]
        pc_df = pd.DataFrame(target_pc_values, columns=pc_columns)

        merged_df = pd.merge(segmentations_df, pc_df, on="Image")

        merged_df["Array"] = merged_df["Image"].str.extract(r"(array\d+)_")

        for array_name, array_df in merged_df.groupby("Array"):
            corr_matrix = pd.DataFrame(index=metric_cols, columns=[f"PC_{n}" for n in target_pc_nums])
            for pc_name in corr_matrix.columns:
                for metric in corr_matrix.index:
                    if metric in array_df.columns:
                        x = array_df[pc_name]
                        y = array_df[metric]
                        corr, _ = pearsonr(x, y)
                        corr_matrix.at[metric, pc_name] = corr

            corr_matrix = corr_matrix.astype(float)

            plt.figure(figsize=(len(corr_matrix.columns) * 1.5 + 4, 8))
            ax = sns.heatmap(
                corr_matrix,
                annot=False,
                cmap="RdBu_r",
                center=0,
                linewidths=0.5,
                square=True,
                cbar=False
            )

            # ax.set_xticks([])
            # ax.set_yticks([])
            # ax.set_xlabel('')
            # ax.set_ylabel('')
            ax.set_title('')
            for spine in ax.spines.values():
                spine.set_visible(False)

            plt.tight_layout()
            plt.show()

In [ ]:
dataset_name = list(all_loaders.keys())[0]
loader = data_loaders[dataset_name]["full"]
matching_samples = []

for idx in loader._test_indices:
    sample_id = loader._get_data_feature(idx, 'sample_id')
    array, raft_str = sample_id.split('_')
    array_num = int(array.replace('array', ''))
    raft_num = int(raft_str)
    if array_num == target_array_num and raft_num == target_raft_id:
        timepoint = int(loader._get_data_feature(idx, 'timepoint'))
        matching_samples.append({'idx': idx, 'timepoint': timepoint, 'sample_id': sample_id})

sorted_samples = sorted(matching_samples, key=lambda x: x['timepoint'])
sorted_indices = [s['idx'] for s in sorted_samples]
sorted_timepoints = [s['timepoint'] for s in sorted_samples]

images = []
for idx in sorted_indices:
    image_group = loader.data[idx]
    image_path = image_group["green"]
    img = plt.imread(image_path)
    images.append(img)

num_images = len(images)
condition = "euploid" if target_array_num == 1 else "anueploid"

if num_images > 0:
    fig, axes = plt.subplots(1, num_images, figsize=(4 * num_images, 4))

    print(f"{condition} | raft {target_raft_id}")

    for i, (img, timepoint) in enumerate(zip(images, sorted_timepoints)):
        plt.imsave(f"figures/{condition}_{target_raft_id}_{i}.png", img, cmap='gray')
        axes[i].imshow(img, cmap='gray')
        axes[i].axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    plt.show()


for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():
        recon_images_current_model = []
        for img in images:
            img_tensor = torch.Tensor(img).unsqueeze(0).unsqueeze(0)
            reconstructed_img, _ = model(img_tensor)
            recon_images_current_model.append(reconstructed_img)

        num_recon_to_plot = len(recon_images_current_model)
        fig, axes = plt.subplots(1, num_recon_to_plot, figsize=(4 * num_recon_to_plot, 4))

        for i, recon_img_tensor in enumerate(recon_images_current_model):
            recon_img = recon_img_tensor.squeeze().detach().numpy()
            plt.imsave(f"figures/{study_name}_reconstructed_{condition}_{target_raft_id}_{i}.png", recon_img, cmap='gray')
            axes[i].imshow(recon_img, cmap='gray')
            axes[i].axis('off')

        fig.suptitle(f'Reconstructed Images - {model_name} | {dataset_name}', fontsize=16, y=0.98)
        plt.tight_layout(rect=[0, 0, 1, 0.96]) 
        plt.show()

fig, axes = plt.subplots(1, num_recon_to_plot, figsize=(4 * num_recon_to_plot, 4))
for i, (orig_img, recon_img_tensor) in enumerate(zip(images, recon_images_current_model)):
    recon_img = recon_img_tensor.squeeze().detach().numpy()

    orig_img_norm = orig_img.astype(np.float32)
    if orig_img_norm.max() > 1.0:
        orig_img_norm /= 255.0

    diff = orig_img_norm - recon_img
    vmax = np.max(np.abs(diff)) 

    # Save diff image
    diff_path = f"figures/{study_name}_diff_{condition}_{target_raft_id}_{i}.png"
    plt.imsave(diff_path, diff, cmap='bwr', vmin=-vmax, vmax=vmax)

    # Show it in the subplot
    axes[i].imshow(diff, cmap='bwr', vmin=-vmax, vmax=vmax)
    axes[i].axis('off')

fig.suptitle(f'Difference Images - {model_name} | {dataset_name}', fontsize=16, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
def generate_spiral_grid(n):
    grid = [[0 for _ in range(n)] for _ in range(n)]
    counter = 1
    for row in range(n - 1, -1, -1):
        if (n - row) % 2 == 1:
            for col in range(0, n):
                grid[row][col] = counter
                counter += 1
        else:
            for col in range(n - 1, -1, -1):
                grid[row][col] = counter
                counter += 1
    edge_plus_two = []
    for i in range(n):
        for j in range(n):
            if (i < 2 or i >= n - 2 or j < 2 or j >= n - 2):
                edge_plus_two.append(grid[i][j])
    return grid, edge_plus_two

rafts = {}
for model_name, dataset in all_models.items():
    rafts[model_name] = {}
    for dataset_name, model in dataset.items():
        rafts[model_name][dataset_name] = []
        loader = data_loaders[dataset_name]["full"]
        for idx in loader._test_indices:
            sample_id = loader._get_data_feature(idx, 'sample_id')
            raft = int(sample_id.split('_')[1])
            rafts[model_name][dataset_name].append(raft)

grid, edge_values = generate_spiral_grid(23)
for model_name, datasets in all_models.items():
    for dataset_name, model in datasets.items():
        pca_embedding = pca_embeddings[model_name][dataset_name]
        rafts_list = rafts[model_name][dataset_name]

        edge_mask = np.array([raft in edge_values for raft in rafts_list])
        edge_points = pca_embedding[edge_mask]
        non_edge_points = pca_embedding[~edge_mask]

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.scatter(edge_points[:, 0], edge_points[:, 1], c='#8B0000', s=15, label='Peripheral')
        ax.scatter(non_edge_points[:, 0], non_edge_points[:, 1], c='gray', s=15, label='Middle')

        ax.set_title(f'{model_name} | {dataset_name}', fontsize=16, y=1.02)
        ax.set_xlabel('PCA Dimension 1')
        ax.set_ylabel('PCA Dimension 2')
        ax.legend(loc='upper right')

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()
